# 🔴 KaizenStat — Advanced Demo (30 min)

**Level:** Advanced | **Time:** ~30 minutes | **Dataset:** Titanic + engineered features

| Topic | What it unlocks |
|-------|-----------------|
| Feature engineering | Better model inputs from domain knowledge |
| `train(tune=True)` | +5–15% score from hyperparameter search |
| `add_model()` | Plug in any sklearn-compatible model |
| `add_check()` | Domain-specific validation rules |
| `feature_impact()` | Counterfactual importance |
| `dataset_difficulty()` | Understand the ceiling score |
| `codegen()` | Export a standalone production script |
| `detect_drift()` | Monitor train vs test distribution |

---
> **Goal:** Get the highest possible Titanic accuracy using every KaizenStat tool available

In [ ]:
!pip install kaizenstat -q
print("✅ KaizenStat installed")

## Setup — Load + Feature Engineering

**Key rule:** Do all feature engineering *before* calling `fit()`.
KaizenStat works on whatever DataFrame you hand it — it never modifies columns you've already built.

We extract meaningful features from the raw columns, then drop the originals.

In [ ]:
import pandas as pd
import numpy as np
from kaizenstat import DataDoctor

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_raw = pd.read_csv(url)
print(f"Raw data: {df_raw.shape}")

df = df_raw.copy()

# 1. Title from Name (before dropping it)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
    'Capt': 'Rare', 'Sir': 'Rare'
}
df['Title'] = df['Title'].map(title_map).fillna('Rare')

# 2. Deck from Cabin (before dropping it)
df['Deck'] = df['Cabin'].str[0].fillna('Unknown')

# 3. Family features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 4. Fare per person
df['FarePerPerson'] = df['Fare'] / df['FamilySize'].replace(0, 1)

# 5. Age groups (fill missing Age before binning)
df['Age'] = df['Age'].fillna(df['Age'].median())
df['AgeGroup'] = pd.cut(
    df['Age'], bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'Adult', 'MidAge', 'Senior']
).astype(str)

# Drop ID / raw text columns — no longer needed
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

print(f"Engineered data: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() > 0 else "None")
df.head()

## 1. Dataset Difficulty

Before tuning, understand the ceiling. 0 = trivially easy, 1 = near-impossible.

In [ ]:
doctor = DataDoctor()
doctor.fit(df, target='Survived')

difficulty = doctor.dataset_difficulty()
print(f"Dataset difficulty: {difficulty:.3f}")
if difficulty > 0.5:
    print("→ Hard. Focus on feature engineering, not tuning.")
elif difficulty > 0.3:
    print("→ Moderate. Tuning will help.")
else:
    print("→ Easy. Any model should work well.")

In [ ]:
health = doctor.health()
print(f"Health Score: {health.score} / 100")

## 2. Custom Validation Checks

Add domain-specific rules that KaizenStat should check on every `validate()` call.

In [ ]:
def check_sex_survival(df, target):
    """Titanic domain rule: female survival rate should be much higher than male."""
    if 'Sex' not in df.columns:
        return ["Missing 'Sex' — strong Titanic predictor"]
    male_rate = df[df['Sex'] == 'male'][target].mean()
    female_rate = df[df['Sex'] == 'female'][target].mean()
    if abs(female_rate - male_rate) < 0.3:
        return [f"Unexpected: sex gap only {abs(female_rate - male_rate):.2f} — check encoding"]
    return []

def check_title_encoded(df, target):
    """Ensure Title feature was extracted."""
    if 'Title' not in df.columns:
        return ["'Title' feature missing — consider extracting from Name for +1–2% AUC"]
    return []

doctor.add_check(check_sex_survival, name="sex_survival")
doctor.add_check(check_title_encoded, name="title_check")

validation = doctor.validate()

## 3. Drift Detection

Simulates what you'd do in production monitoring — check if new data distributions match training.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Survived'])
y = df['Survived']
X_train, X_test, _, _ = train_test_split(X, y, test_size=0.2, random_state=42)

# Drift detection on numeric columns only
num_cols = X_train.select_dtypes(include='number').columns.tolist()
drift = doctor.detect_drift(X_train[num_cols], X_test[num_cols])

print("Drift detection (features with p < 0.05 are drifting):")
if drift:
    for col, pval in sorted(drift.items(), key=lambda x: x[1]):
        flag = " ⚠️ DRIFT" if pval < 0.05 else ""
        print(f"  {col:20s}  p={pval:.4f}{flag}")
else:
    print("  ✅ No significant drift detected")

In [ ]:
fixed_df = doctor.fix(safe=True)
print(f"Fixed: {fixed_df.isnull().sum().sum()} missing values remaining")

## 4. Custom Model — Compete Against Built-ins

`add_model()` injects any sklearn-compatible estimator into the benchmark.

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier

doctor.add_model("SVM_RBF",    SVC(kernel='rbf', probability=True, C=1.0, random_state=42))
doctor.add_model("ExtraTrees", ExtraTreesClassifier(n_estimators=100, random_state=42))

print("Custom models registered — they compete in the next train() call.")

## 5. Train with Hyperparameter Tuning

`tune=True` runs `RandomizedSearchCV` on the winner from the benchmark.
Typical gain: +3–10% over untuned.

In [ ]:
print("Training with tuning (~2–3 min)...")
train_result = doctor.train(cv=5, tune=True, n_iter=30)

print(f"\n{'='*50}")
print(f"Best model:  {train_result.model_name}")
print(f"Test score:  {train_result.test_score:.4f}")
print(f"Train score: {train_result.train_score:.4f}")
if hasattr(train_result, 'best_params') and train_result.best_params:
    print(f"Best params: {train_result.best_params}")

## 6. Compare: Baseline vs Engineered + Tuned

In [ ]:
# Baseline: raw data, no tuning
df_raw_clean = pd.read_csv(url)
df_raw_clean = df_raw_clean.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

d_base = DataDoctor()
d_base.fit(df_raw_clean, target='Survived')
d_base.fix(safe=True)
base = d_base.train(cv=5, tune=False)

print(f"\n{'='*50}")
print("COMPARISON")
print(f"{'='*50}")
print(f"Baseline (raw, no tuning):       {base.test_score:.4f}")
print(f"Engineered + tuned:              {train_result.test_score:.4f}")
delta = train_result.test_score - base.test_score
print(f"Improvement from FE + tuning:    {delta:+.4f} ({delta*100:.1f}%)")

In [ ]:
debug_result = doctor.debug_model()

print(f"\nTest score: {debug_result.test_score:.4f}")
print(f"Gap:        {debug_result.gap:.4f}")

In [ ]:
print("Counterfactual Feature Impact:\n")
impact = doctor.feature_impact(top_n=15)
for feat, drop in sorted(impact.items(), key=lambda x: -x[1]):
    bar = '█' * max(1, int(drop * 200))
    print(f"  {feat:25s}  {drop:.4f}  {bar}")

print("\nInsight: if Title or Sex is top → domain features helped.")
print("If FamilySize or FarePerPerson → feature engineering paid off.")

In [ ]:
improvement_report = doctor.improve()

print("\n--- Recommended actions ---")
for i, action in enumerate(doctor.recommend_actions(), 1):
    print(f"  {i}. {action}")

In [ ]:
trust = doctor.trust_score()
confidence = doctor.pipeline_confidence()

print(f"Trust Score:         {trust.score:.0f} / 100")
print(f"Pipeline Confidence: {confidence} / 100")
if trust.score >= 75:
    print("✅ PRODUCTION READY")
else:
    print("🟡 Needs improvement before deploying")

## 7. Code Generation — Export Production Script

`codegen()` exports a **standalone sklearn script** — no KaizenStat dependency at runtime.

In [ ]:
script_path = doctor.codegen(output_path="titanic_pipeline.py")
print(f"✅ Standalone script: {script_path}")

with open(script_path, 'r') as f:
    code = f.read()

print(f"\n--- Preview (first 30 lines) ---")
for i, line in enumerate(code.split('\n')[:30], 1):
    print(f"{i:3d}  {line}")

In [ ]:
# Export trained model
model_path = doctor.export_model(path="titanic_advanced_model.joblib")
print(f"✅ Model exported: {model_path}")

# Full report
report_path = doctor.report(output_path="advanced_report.html")
print(f"✅ Report: {report_path}")

from IPython.display import IFrame, display
display(IFrame(src='advanced_report.html', width='100%', height='700px'))

## 8. Production Inference

Load the exported model and predict on new passengers — simulates what you'd do in a deployed API.

In [ ]:
import joblib

pipeline = joblib.load(model_path)
print(f"Loaded: {type(pipeline).__name__}")

# Sample passengers — must have same columns as training data (after FE, before fit)
samples = pd.DataFrame([
    # 1st class woman — historically very high survival rate
    {'Pclass': 1, 'Sex': 'female', 'Age': 35.0, 'SibSp': 1, 'Parch': 0,
     'Fare': 80.0, 'Embarked': 'S', 'Title': 'Mrs',
     'Deck': 'C', 'FamilySize': 2, 'IsAlone': 0,
     'FarePerPerson': 40.0, 'AgeGroup': 'Adult'},
    # 3rd class man alone — historically low survival rate
    {'Pclass': 3, 'Sex': 'male', 'Age': 22.0, 'SibSp': 0, 'Parch': 0,
     'Fare': 7.5, 'Embarked': 'S', 'Title': 'Mr',
     'Deck': 'Unknown', 'FamilySize': 1, 'IsAlone': 1,
     'FarePerPerson': 7.5, 'AgeGroup': 'Adult'},
])

preds = pipeline.predict(samples)
for i, pred in enumerate(preds):
    label = 'SURVIVED' if pred == 1 else 'DID NOT SURVIVE'
    sex   = samples.iloc[i]['Sex']
    cls   = samples.iloc[i]['Pclass']
    print(f"  Passenger {i+1} (Class {cls}, {sex}): {label}")

## Summary — Everything We Used

```python
# Feature engineer BEFORE fit()
df['Title']         = ...  # extracted from Name
df['Deck']          = ...  # extracted from Cabin
df['FamilySize']    = df['SibSp'] + df['Parch'] + 1
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
df = df.drop(columns=['PassengerId','Name','Ticket','Cabin'])

doctor = DataDoctor()
doctor.fit(df, target='Survived')
doctor.dataset_difficulty()                # 0–1 ceiling score
doctor.health()                            # data quality score
doctor.add_check(fn, name='...')           # domain rules
doctor.validate()                          # + leakage + drift + custom
doctor.detect_drift(X_train, X_test)       # distribution shift
doctor.fix(safe=True)                      # auto-heal
doctor.add_model('SVM', SVC(...))          # custom competitor
doctor.train(tune=True, n_iter=30)         # benchmark + tune
doctor.debug_model()                       # root-cause analysis
doctor.feature_impact(top_n=15)           # counterfactual importance
doctor.improve()                           # ranked suggestions
doctor.recommend_actions()                 # structured next steps
doctor.trust_score()                       # production readiness
doctor.pipeline_confidence()               # holistic score
doctor.codegen(output_path='pipeline.py') # standalone production script
doctor.export_model(path='model.joblib')  # save trained model
doctor.report()                            # full HTML report
```

## 🎯 Further Challenges

**Challenge 1:** Can you break 85% accuracy?
```python
# Try interaction features:
df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)
df['Age_Pclass'] = df['Age'] * df['Pclass']
```

**Challenge 2:** Use `auto_improve()` with tuning
```python
doctor_auto = DataDoctor()
doctor_auto.fit(df, target='Survived')
comparison = doctor_auto.auto_improve(tune=True)
print(f"Delta: {comparison.score_delta:+.4f}")
```

**Challenge 3:** Try a different Kaggle dataset
- [House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) — regression
- [Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — imbalanced classification

---
## Notebook Series

| | Level | Time |
|-|-------|------|
| [Basic](demo_basic.ipynb) | 🟢 | 5 min |
| [Intermediate](demo_intermediate.ipynb) | 🟡 | 15 min |
| **You are here** | 🔴 | 30 min |

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/kaizenstat-python/KaizenStat) · MIT License*